# Environment Setup — UNeXt (`unext`)

This notebook creates and configures the conda `unext` environment needed to run:
- `01_preprocessing_BUSI.ipynb`
- `02_train_inference_UNeXt_BUSI.ipynb`
- `01_preprocessing_ISIC.ipynb`
- `02_train_inference_UNeXt_ISIC.ipynb`

**Run this notebook once** with the default kernel before using the others.

---

| Step | Description |
|:----:|:------------|
| 1 | Create the conda `unext` environment with Python 3.6 |
| 2 | Install PyTorch 1.9.1 + CUDA 11.1 (RTX 30xx compatible) |
| 3 | Install project dependencies (timm, albumentations, etc.) |
| 4 | Register the Jupyter kernel |
| 5 | Verify imports and GPU access |

In [ ]:
import os
from pathlib import Path

CONDA_BIN    = Path('/home/bruno-borges/miniconda3/bin/conda')
PIP_BIN      = Path('/home/bruno-borges/miniconda3/envs/unext/bin/pip')
PYTHON_BIN   = Path('/home/bruno-borges/miniconda3/envs/unext/bin/python')
ENV_NAME     = 'unext'
PYTHON_VER   = '3.6'

checks = {
    'conda binary': CONDA_BIN,
}
all_ok = True
for name, path in checks.items():
    status = '✓' if path.exists() else '✗ NOT FOUND'
    print(f'  [{status}] {name}: {path}')
    if not path.exists():
        all_ok = False

if not all_ok:
    print('\nFix the missing paths before proceeding.')
else:
    print('\nPaths OK — proceed to Step 1.')

## 1. Create the conda `unext` environment

Creates the environment with **Python 3.6** if it does not already exist or uses an incompatible Python version.

> Python 3.6 is required for compatibility with the PyTorch 1.9.1+cu111 wheels used here.

In [ ]:
%%bash
set -e

CONDA=/home/bruno-borges/miniconda3/bin/conda
ENV_NAME=unext
PYTHON_VER=3.6
PYTHON_BIN=/home/bruno-borges/miniconda3/envs/${ENV_NAME}/bin/python

NEEDS_RECREATE=true
if [ -x "$PYTHON_BIN" ]; then
    MAJOR=$($PYTHON_BIN -c "import sys; print(sys.version_info.major)")
    MINOR=$($PYTHON_BIN -c "import sys; print(sys.version_info.minor)")
    if [ "$MAJOR" -eq 3 ] && [ "$MINOR" -eq 6 ]; then
        echo "[INFO] Environment ${ENV_NAME} already uses Python $($PYTHON_BIN --version) — nothing to do."
        NEEDS_RECREATE=false
    else
        echo "[INFO] Environment ${ENV_NAME} uses Python $($PYTHON_BIN --version) — needs to be recreated with Python ${PYTHON_VER}."
    fi
fi

if [ "$NEEDS_RECREATE" = true ]; then
    echo "[INFO] Creating environment ${ENV_NAME} with Python ${PYTHON_VER}..."
    $CONDA create -n ${ENV_NAME} python=${PYTHON_VER} -y
    echo "✓ Environment ${ENV_NAME} created."
fi

## 2. Install PyTorch 1.9.1 + CUDA 11.1

PyTorch 1.9.1 is the **last release supporting Python 3.6** and the **first to include sm_86** (RTX 30xx / Ampere).  
The CUDA 11.1 wheel runs correctly under the system's CUDA 12.x driver (backward compatible).

In [ ]:
%%bash
set -e

PIP=/home/bruno-borges/miniconda3/envs/unext/bin/pip

echo "[INFO] Installing PyTorch 1.9.1 + CUDA 11.1..."
$PIP install \
    torch==1.9.1+cu111 \
    torchvision==0.10.1+cu111 \
    -f https://download.pytorch.org/whl/torch_stable.html \
    --quiet
echo "✓ PyTorch + torchvision installed."

## 3. Install project dependencies

Pins all packages to the versions present in the working environment.

In [ ]:
%%bash
set -e

PIP=/home/bruno-borges/miniconda3/envs/unext/bin/pip

echo "[INFO] Installing core numerical and image packages..."
$PIP install \
    numpy==1.19.5 \
    scipy==1.5.4 \
    scikit-image==0.17.2 \
    scikit-learn==0.24.2 \
    Pillow==8.4.0 \
    opencv-python==4.5.1.48 \
    --quiet
echo "✓ Core packages installed."

echo "[INFO] Installing deep-learning utilities..."
$PIP install \
    timm==0.4.12 \
    albumentations==1.0.3 \
    tqdm==4.64.1 \
    --quiet
echo "✓ DL utilities installed."

echo "[INFO] Installing data / visualisation packages..."
$PIP install \
    pandas==1.1.5 \
    matplotlib==3.3.4 \
    --quiet
echo "✓ Data/visualisation packages installed."

echo "[INFO] Installing Python 3.6 backports..."
$PIP install \
    dataclasses==0.8 \
    typing_extensions==4.0.0 \
    --quiet
echo "✓ Backports installed."

## 4. Register the Jupyter kernel

Registers the environment as a selectable kernel in Jupyter Lab / Notebook.

In [ ]:
%%bash
set -e

PYTHON=/home/bruno-borges/miniconda3/envs/unext/bin/python

$PYTHON -m pip install ipykernel==5.5.6 --quiet

$PYTHON -m ipykernel install --user --name=unext --display-name='Python (unext)'

echo ""
echo "✓ Kernel 'unext' registered."
echo "  Select 'Python (unext)' when running the BUSI and ISIC notebooks."

## 5. Import verification

In [ ]:
%%bash

PYTHON=/home/bruno-borges/miniconda3/envs/unext/bin/python

echo "=== Import verification in unext env ==="
echo ""

packages=(
    "torch"
    "torchvision"
    "timm"
    "albumentations"
    "cv2"
    "numpy"
    "scipy"
    "skimage"
    "sklearn"
    "PIL"
    "pandas"
    "matplotlib"
    "tqdm"
)

all_ok=true
for pkg in "${packages[@]}"; do
    if $PYTHON -c "import $pkg" 2>/dev/null; then
        version=$($PYTHON -c "import $pkg; print(getattr($pkg, '__version__', 'n/a'))" 2>/dev/null)
        echo "  [✓] $pkg  $version"
    else
        echo "  [✗] $pkg  — IMPORT FAILED"
        all_ok=false
    fi
done

echo ""
if $all_ok; then
    echo "✓ All imports verified."
else
    echo "✗ Some imports failed — re-run the installation cells above."
fi

In [ ]:
%%bash

PYTHON=/home/bruno-borges/miniconda3/envs/unext/bin/python

echo "=== PyTorch — GPU ==="
$PYTHON - <<'EOF'
import torch
print(f'  torch version  : {torch.__version__}')
print(f'  CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'  GPU            : {torch.cuda.get_device_name(0)}')
    print(f'  Capability     : sm_{cap[0]}{cap[1]}')
    print(f'  CUDA runtime   : {torch.version.cuda}')
    x = torch.randn(4, 4).cuda()
    print(f'  Tensor on GPU  : OK  (sum={x.sum().item():.4f})')
EOF

## Summary

```
Conda environment : unext  (Python 3.6)
Jupyter kernel    : Python (unext)

Key packages:
  torch           1.9.1+cu111   ← sm_86 (RTX 30xx) support, last py3.6 release
  torchvision     0.10.1+cu111
  timm            0.4.12        ← compatible with torch 1.9 (no torch._six)
  albumentations  1.0.3
  numpy           1.19.5
  opencv-python   4.5.1.48
  pandas          1.1.5
  scikit-image    0.17.2
  scikit-learn    0.24.2
  scipy           1.5.4
  matplotlib      3.3.4
  tqdm            4.64.1
```

### Next steps

1. Select **Python (unext)** as the kernel in:
   - `01_preprocessing_BUSI.ipynb`
   - `02_train_inference_UNeXt_BUSI.ipynb`
   - `01_preprocessing_ISIC.ipynb`
   - `02_train_inference_UNeXt_ISIC.ipynb`
2. Run `01_preprocessing_*.ipynb` to generate the `inputs/` directories.
3. Run `02_train_inference_*.ipynb` to train the model and export predictions.